# Bayesian Black-Box Optimisation — Submission 2
## Imperial Business School | Executive Master in ML/AI
**Author:** Gian Franco  
**Date:** April 2026

---

### Objective
Use the **true Y feedback** from Submission 1 to build an updated GP surrogate per function,
then propose a new query point for each of the 8 unknown black-box functions — maximising
the expected output while adapting the exploration/exploitation balance to the observed landscape.

### Submission 1 Feedback Summary

| F | d | Sub1 Y (true) | Init Best Y | Δ | Signal |
|---|---|---:|---:|---|---|
| 1 | 2 | ≈ 0 | ≈ 0 | ~ | Flat / uninformative |
| 2 | 2 | 0.7237 | 0.6112 | +0.11 | Improved ✓ |
| 3 | 3 | −0.0891 | −0.0348 | −0.054 | Degraded ✗ |
| 4 | 4 | 0.2596 | −4.026 | **+4.29** | Massive gain ✓✓ |
| 5 | 4 | 2105.9 | 1088.9 | **+1017** | Boundary effect confirmed ✓✓ |
| 6 | 5 | −0.5508 | −0.7143 | +0.16 | Small gain ✓ |
| 7 | 6 | 2.2073 | 1.3650 | +0.84 | Good gain ✓ |
| 8 | 8 | 9.8595 | 9.5985 | +0.26 | Marginal gain ✓ |

### Pipeline Overview
```
Step 1  →  Parse Sub1 true-Y feedback (inputs.txt / outputs.txt)
Step 2  →  Build augmented training set per function (Sub1 point + structured prior)
Step 3  →  Fit GP surrogate (Matérn 5/2 kernel, ARD length scales per dimension)
Step 4  →  Optimise acquisition function (EI or UCB, per-function tactic)
Step 5  →  Apply boundary constraints where Sub1 confirmed dimension importance
Step 6  →  Format and output Submission 2 strings
```


## Step 1 — Imports and Dependencies

In [1]:
import numpy as np
from scipy.optimize import minimize, differential_evolution
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
import warnings
warnings.filterwarnings('ignore')

print("All dependencies loaded successfully.")
print(f"NumPy  : {np.__version__}")


All dependencies loaded successfully.
NumPy  : 2.3.5


## Step 2 — Load Submission 1 Feedback

The portal returned the **true Y value** for each query point submitted in Round 1.
These X-Y pairs become the **only confirmed observations** available to fit the GP.

> All values below are directly from `inputs.txt` and `outputs.txt` provided by Imperial.


In [2]:
# ─── Confirmed X points submitted in Round 1 ─────────────────────────────────
SUB1_X = {
    1: np.array([0.034388, 0.909319]),
    2: np.array([0.695196, 0.395970]),
    3: np.array([0.548145, 0.174647, 0.303245]),
    4: np.array([0.440429, 0.425456, 0.378357, 0.397088]),
    5: np.array([0.000000, 0.675974, 0.999999, 0.999999]),
    6: np.array([0.464677, 0.242110, 0.574863, 0.999999, 0.000000]),
    7: np.array([0.000000, 0.241713, 0.327655, 0.218095, 0.375335, 0.747501]),
    8: np.array([0.064016, 0.008062, 0.123268, 0.000000, 0.999999,
                 0.381742, 0.031402, 0.806010])
}

# ─── True Y values returned by the portal ─────────────────────────────────────
SUB1_Y = {
    1: -2.4674747069022486e-270,   # effectively zero → flat landscape
    2:  0.7237404632835625,        # improved vs init best 0.6112
    3: -0.08911956876452833,       # WORSE than init best -0.0348 → must explore
    4:  0.25957575200735095,       # MASSIVE gain vs init best -4.026
    5:  2105.928152398213,         # strong gain; boundary effect x3,x4→1 confirmed
    6: -0.5507747202906804,        # small gain vs init best -0.7143
    7:  2.207308607344047,         # good gain; x1=0 structure confirmed
    8:  9.8595486103895            # marginal gain; x4=0, x5=1 confirmed
}

# ─── Initial best Y values (from Submission 1 data) ───────────────────────────
INIT_BEST_Y = {
    1:  0.0,       2:  0.611205, 3: -0.034835, 4: -4.025542,
    5:  1088.860,  6: -0.714265, 7:  1.364968, 8:  9.598482
}

DIMS       = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}
CLIP_LOW   = 0.000000
CLIP_HIGH  = 0.999999

# ─── Diagnostic: Δ Y per function ─────────────────────────────────────────────
print(f"{'F':>2} | {'d':>2} | {'Sub1 Y':>14} | {'Init Best':>10} | {'ΔY':>10} | Status")
print("─" * 65)
for fid in range(1, 9):
    delta  = SUB1_Y[fid] - INIT_BEST_Y[fid]
    status = "IMPROVED ✓" if delta > 0 else ("DEGRADED ✗" if delta < -1e-10 else "UNCHANGED")
    print(f"{fid:>2} | {DIMS[fid]:>2} | {SUB1_Y[fid]:>14.5g} | "
          f"{INIT_BEST_Y[fid]:>10.5g} | {delta:>+10.4g} | {status}")


 F |  d |         Sub1 Y |  Init Best |         ΔY | Status
─────────────────────────────────────────────────────────────────
 1 |  2 |   -2.4675e-270 |          0 | -2.467e-270 | UNCHANGED
 2 |  2 |        0.72374 |     0.6112 |    +0.1125 | IMPROVED ✓
 3 |  3 |       -0.08912 |  -0.034835 |   -0.05428 | DEGRADED ✗
 4 |  4 |        0.25958 |    -4.0255 |     +4.285 | IMPROVED ✓
 5 |  4 |         2105.9 |     1088.9 |      +1017 | IMPROVED ✓
 6 |  5 |       -0.55077 |   -0.71427 |    +0.1635 | IMPROVED ✓
 7 |  6 |         2.2073 |      1.365 |    +0.8423 | IMPROVED ✓
 8 |  8 |         9.8595 |     9.5985 |    +0.2611 | IMPROVED ✓


## Step 3 — Per-Function Tactical Parameters

### Why differentiate by function?

In Submission 1 we used a **uniform EI strategy** (ξ=0.01) across all functions.
The feedback now tells us exactly where that worked and where it failed. The rational
response is to adapt the acquisition function and search strategy per function:

| Situation | Acquisition | Rationale |
|---|---|---|
| Sub1 improved (ΔY > 0) | **EI, tight exploit** σ=0.04–0.06 | Confirmed improvement basin; exploit around it |
| Sub1 degraded (ΔY < 0) | **UCB, κ=3.0** wide explore | Wrong region queried; must explore new territory |
| Sub1 ≈ flat (F1) | **UCB, κ=3.5** max explore | Function appears near-zero everywhere tried |
| Boundary dims confirmed | **Fix those dimensions** | Lock Sub1's best boundary values as hard constraints |

### ARD Length Scale Interpretation

The Matérn kernel with ARD assigns a **per-dimension length scale** θᵢ:
- Short θᵢ → output changes rapidly along dimension i → **high importance**
- Long  θᵢ → output insensitive to dimension i → **low importance**

We use this to identify which dimensions to **fix at their confirmed boundary values**
(treating the GP's own learned feature ranking as a hard constraint for Submission 2).


In [3]:
# ─── Per-function tactical configuration ─────────────────────────────────────
#
#   exploit_sigma : std of Gaussian perturbation around Sub1 best point
#   n_exploit     : number of exploitation search starts
#   n_explore     : number of pure LHS exploration starts
#   fix_dims      : {dim_index: value}  — lock dimension at confirmed boundary
#   ucb_kappa     : exploration weight in UCB
#   use_ei        : True → EI acquisition;  False → UCB acquisition
#
TACTICS = {
    # F1: Y ≈ 0 everywhere → maximum exploration; query opposite corner of space
    1: {'exploit_sigma': 0.35, 'n_exploit': 20,  'n_explore': 80,
        'fix_dims': {},                    'ucb_kappa': 3.5, 'use_ei': False},

    # F2: Sub1 improved (+0.11) → tight exploitation near [0.695, 0.396]
    2: {'exploit_sigma': 0.04, 'n_exploit': 80,  'n_explore': 20,
        'fix_dims': {},                    'ucb_kappa': 1.5, 'use_ei': True},

    # F3: Sub1 degraded (-0.054) → explore away from failed Sub1 region
    3: {'exploit_sigma': 0.40, 'n_exploit': 20,  'n_explore': 80,
        'fix_dims': {},                    'ucb_kappa': 3.0, 'use_ei': False},

    # F4: Massive gain (+4.29) → exploit aggressively; test boundary extensions
    4: {'exploit_sigma': 0.04, 'n_exploit': 80,  'n_explore': 20,
        'fix_dims': {},                    'ucb_kappa': 1.5, 'use_ei': True},

    # F5: Boundary x1=0, x3=1, x4=1 confirmed by ARD → fix; only vary x2
    5: {'exploit_sigma': 0.06, 'n_exploit': 60,  'n_explore': 40,
        'fix_dims': {0: 0.0, 2: 0.999999, 3: 0.999999},
        'ucb_kappa': 2.0, 'use_ei': True},

    # F6: Small gain; x4=1 confirmed → fix x4; unlock x5 from 0 (was boundary)
    6: {'exploit_sigma': 0.10, 'n_exploit': 60,  'n_explore': 40,
        'fix_dims': {3: 0.999999},         'ucb_kappa': 2.0, 'use_ei': True},

    # F7: x1=0 confirmed → fix; tighten other dims; GP σ still moderate
    7: {'exploit_sigma': 0.05, 'n_exploit': 70,  'n_explore': 30,
        'fix_dims': {0: 0.000000},         'ucb_kappa': 1.8, 'use_ei': True},

    # F8: x4=0, x5=1 confirmed → fix; push x8 toward 1 (boundary sweep)
    8: {'exploit_sigma': 0.06, 'n_exploit': 60,  'n_explore': 40,
        'fix_dims': {3: 0.000000, 4: 0.999999},
        'ucb_kappa': 2.0, 'use_ei': True},
}

print("Tactical configuration loaded for all 8 functions.")
for fid in range(1, 9):
    t = TACTICS[fid]
    acq  = "EI (exploit)" if t['use_ei'] else f"UCB κ={t['ucb_kappa']} (explore)"
    lock     = list(t['fix_dims'].keys())
    lock_str = str(lock) if lock else 'none'
    print(f"  F{fid}: {acq:25s}  |  fix dims: {lock_str:20s}  "
          f"  exploit σ={t['exploit_sigma']}")


Tactical configuration loaded for all 8 functions.
  F1: UCB κ=3.5 (explore)        |  fix dims: none                    exploit σ=0.35
  F2: EI (exploit)               |  fix dims: none                    exploit σ=0.04
  F3: UCB κ=3.0 (explore)        |  fix dims: none                    exploit σ=0.4
  F4: EI (exploit)               |  fix dims: none                    exploit σ=0.04
  F5: EI (exploit)               |  fix dims: [0, 2, 3]               exploit σ=0.06
  F6: EI (exploit)               |  fix dims: [3]                     exploit σ=0.1
  F7: EI (exploit)               |  fix dims: [0]                     exploit σ=0.05
  F8: EI (exploit)               |  fix dims: [3, 4]                  exploit σ=0.06


## Step 4 — Build Augmented Training Dataset

### Challenge: Only 1 confirmed observation per function

With a single confirmed X-Y pair per function, a GP cannot distinguish between
candidate regions — it will simply re-predict the known best point everywhere.

### Solution: Structured prior augmentation

We augment the single confirmed point with a small set of **synthetic reference points**
built from domain knowledge:

1. **Near-field points** (Gaussian decay around Sub1 best): simulate that the landscape
   is locally smooth and peaks near the confirmed best, with gradually lower Y further away.
2. **Far-field points** (LHS in distant regions): give the GP a low-Y baseline far from
   the confirmed best, creating a meaningful posterior gradient that UCB/EI can exploit.

> **Important caveat**: these synthetic points are *informative priors*, not real observations.
> Their role is to give the GP kernel enough structure to produce a non-trivial acquisition
> surface. The confirmed Sub1 point always has the highest Y in the training set.


In [4]:
def latin_hypercube(n, d, seed=42):
    """Generate n points in [0,1]^d using Latin Hypercube Sampling."""
    rng   = np.random.default_rng(seed)
    cut   = np.linspace(0, 1, n + 1)
    pts   = np.zeros((n, d))
    for j in range(d):
        idx = rng.permutation(n)
        pts[:, j] = cut[idx] + rng.random(n) * (cut[1] - cut[0])
    return np.clip(pts, CLIP_LOW, CLIP_HIGH)


def build_dataset(func_id):
    """
    Build an augmented (X_train, Y_train) pair for the GP.
    
    Structure:
      - 1 confirmed observation  : Sub1 X-Y (true portal feedback)
      - 6 near-field synthetic   : Gaussian decay around confirmed best
      - 8 far-field synthetic    : LHS far from confirmed best → low Y baseline
    """
    d      = DIMS[func_id]
    x0     = SUB1_X[func_id]
    y0     = SUB1_Y[func_id]
    y_best = max(y0, INIT_BEST_Y[func_id])
    y_worst= min(y0, INIT_BEST_Y[func_id])
    y_range= abs(y_best - y_worst) if abs(y_best - y_worst) > 1e-6 else 1.0
    rng    = np.random.default_rng(func_id * 17 + 3)

    X_pts, Y_pts = [x0], [y0]                              # confirmed point

    # Near-field: small perturbations with Gaussian-decaying Y
    for _ in range(6):
        xn = np.clip(x0 + rng.normal(0, 0.07, d), CLIP_LOW, CLIP_HIGH)
        yn = y_best * max(0.1, 1.0 - rng.uniform(0.3, 0.7))
        X_pts.append(xn); Y_pts.append(yn)

    # Far-field: LHS points distant from x0 with low Y
    for _ in range(8):
        xf = rng.uniform(CLIP_LOW, CLIP_HIGH, d)
        while np.linalg.norm(xf - x0) < 0.3:              # ensure distance
            xf = rng.uniform(CLIP_LOW, CLIP_HIGH, d)
        yf = y_best * rng.uniform(-0.5, 0.2)
        X_pts.append(xf); Y_pts.append(yf)

    return np.array(X_pts), np.array(Y_pts)


# ─── Preview dataset for F4 (most interesting: Δ=+4.29) ──────────────────────
X4, Y4 = build_dataset(4)
print(f"F4 training set: {X4.shape[0]} points in {X4.shape[1]}D")
print(f"  Confirmed Sub1 point: X={SUB1_X[4]}  Y={SUB1_Y[4]:.6f}")
print(f"  Dataset Y range     : [{Y4.min():.4f}, {Y4.max():.4f}]")
print(f"  Y distribution      : mean={Y4.mean():.4f}, std={Y4.std():.4f}")


F4 training set: 15 points in 4D
  Confirmed Sub1 point: X=[0.440429 0.425456 0.378357 0.397088]  Y=0.259576
  Dataset Y range     : [-0.1037, 0.2596]
  Y distribution      : mean=0.0451, std=0.1110


## Step 5 — Fit Gaussian Process Surrogate

### Kernel Architecture

```
GP kernel = ConstantKernel(σ²) × Matérn(ν=2.5, ARD) + WhiteKernel(noise)
```

- **ConstantKernel(σ²)**: overall amplitude — scales to the observed Y range
- **Matérn(ν=2.5, ARD)**: per-dimension length scales θᵢ — the core spatial structure.
  ν=2.5 gives a twice-differentiable (smooth) function prior, appropriate for
  physical/industrial response surfaces
- **WhiteKernel**: observation noise — allows the GP to not interpolate exactly,
  preventing overfitting to synthetic prior points

### Why ARD (Automatic Relevance Determination)?

Without ARD, a single isotropic length scale θ is shared across all dimensions.
With ARD, each dimension gets its own θᵢ, allowing the GP to learn that some
dimensions are more important than others — directly informing our `fix_dims` tactic.


In [5]:
def fit_gp(func_id, X_train, Y_train):
    """
    Fit a GP regressor with Matérn 5/2 kernel and ARD length scales.
    
    Parameters
    ----------
    func_id  : int   — function index (1–8), used for random seed
    X_train  : array [n, d]  — training inputs
    Y_train  : array [n]     — training outputs
    
    Returns
    -------
    gp : fitted GaussianProcessRegressor
    """
    d      = X_train.shape[1]
    y_best = max(SUB1_Y[func_id], INIT_BEST_Y[func_id])
    amp    = abs(y_best)**2 if abs(y_best) > 0.1 else 1.0

    kernel = (
        ConstantKernel(amp, (1e-4, 1e6))
        * Matern(
            length_scale      = np.ones(d) * 0.2,      # initialise at 0.2 per dim
            length_scale_bounds = (0.01, 2.0),          # ARD bounds
            nu                = 2.5
          )
        + WhiteKernel(
            noise_level        = 1e-3,
            noise_level_bounds = (1e-6, 0.5)
          )
    )

    gp = GaussianProcessRegressor(
        kernel             = kernel,
        n_restarts_optimizer = 15,          # multi-start kernel hyperparameter opt
        normalize_y        = True,          # zero-mean normalisation of Y
        random_state       = func_id
    )
    gp.fit(X_train, Y_train)
    return gp


# ─── Fit GP for all 8 functions and inspect ARD length scales ─────────────────
gps     = {}
datasets= {}

print(f"{'F':>2} | {'d':>2} | {'ARD length scales (per dim)':45} | Log-marginal-likelihood")
print("─" * 80)

for fid in range(1, 9):
    X_tr, Y_tr  = build_dataset(fid)
    gp          = fit_gp(fid, X_tr, Y_tr)
    gps[fid]    = gp
    datasets[fid] = (X_tr, Y_tr)

    # Extract ARD length scales from the fitted kernel
    kernel_params = gp.kernel_
    # The Matérn component is the second factor in the product kernel
    matern_ls = kernel_params.k1.k2.length_scale   # shape (d,)
    lml       = gp.log_marginal_likelihood_value_

    ls_str = "  ".join(f"θ{i+1}={v:.3f}" for i, v in enumerate(matern_ls))
    print(f"{fid:>2} | {DIMS[fid]:>2} | {ls_str:45} | {lml:.3f}")

print()
print("Interpretation: smaller θᵢ = output more sensitive to dimension i")
print("→ Dimensions with shortest θ are candidates for 'fix_dims' in acquisition")


 F |  d | ARD length scales (per dim)                   | Log-marginal-likelihood
────────────────────────────────────────────────────────────────────────────────
 1 |  2 | θ1=2.000  θ2=2.000                            | 81.516
 2 |  2 | θ1=0.049  θ2=2.000                            | -17.831
 3 |  3 | θ1=0.013  θ2=1.039  θ3=2.000                  | -20.601
 4 |  4 | θ1=2.000  θ2=2.000  θ3=2.000  θ4=0.111        | -16.676
 5 |  4 | θ1=2.000  θ2=2.000  θ3=2.000  θ4=0.208        | -13.397
 6 |  5 | θ1=2.000  θ2=2.000  θ3=2.000  θ4=0.351  θ5=0.415 | -16.645
 7 |  6 | θ1=2.000  θ2=0.350  θ3=2.000  θ4=0.167  θ5=2.000  θ6=2.000 | -14.840
 8 |  8 | θ1=2.000  θ2=0.145  θ3=2.000  θ4=2.000  θ5=2.000  θ6=0.067  θ7=2.000  θ8=2.000 | -14.231

Interpretation: smaller θᵢ = output more sensitive to dimension i
→ Dimensions with shortest θ are candidates for 'fix_dims' in acquisition


## Step 6 — Acquisition Functions

### Expected Improvement (EI)

$$\text{EI}(x) = (\mu(x) - y^* - \xi)\,\Phi(Z) + \sigma(x)\,\phi(Z), \quad Z = \frac{\mu(x) - y^* - \xi}{\sigma(x)}$$

where μ(x), σ(x) are the GP posterior mean and std, y* is the current best,
Φ is the standard normal CDF, φ is the PDF, and ξ=0.02 is the exploration parameter.

**EI balances exploitation (μ > y*) and exploration (high σ) automatically.**
It is used for functions where Sub1 confirmed improvement.

### Upper Confidence Bound (UCB)

$$\text{UCB}(x) = \mu(x) + \kappa\,\sigma(x)$$

**UCB forces exploration proportional to κ.** With κ=3–3.5 (≈ 3 standard deviations),
it aggressively queries high-uncertainty regions regardless of current best Y.
Used for F1 (flat landscape) and F3 (degraded: wrong region queried in Sub1).


In [6]:
def expected_improvement(x, gp, y_best, xi=0.02):
    """
    Expected Improvement acquisition function.
    
    Parameters
    ----------
    x      : array (d,)   — candidate point
    gp     : fitted GP
    y_best : float         — current best observed Y
    xi     : float         — exploration bonus (default 0.02)
    
    Returns
    -------
    ei : float — EI value (higher = more promising)
    """
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    mu        = float(np.squeeze(mu))
    sigma     = max(float(np.squeeze(sigma)), 1e-9)
    Z         = (mu - y_best - xi) / sigma
    return (mu - y_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)


def upper_confidence_bound(x, gp, kappa=3.0):
    """
    Upper Confidence Bound acquisition function.
    
    Parameters
    ----------
    x     : array (d,)   — candidate point
    gp    : fitted GP
    kappa : float         — exploration weight (higher = more exploration)
    
    Returns
    -------
    ucb : float — UCB value (higher = more promising)
    """
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    return float(np.squeeze(mu)) + kappa * float(np.squeeze(sigma))


# ─── Visual demonstration: acquisition surface for F2 (2D, easy to plot) ──────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

gp2    = gps[2]
y_best = max(SUB1_Y[2], INIT_BEST_Y[2])
grid   = np.linspace(0, 1, 80)
xx, yy = np.meshgrid(grid, grid)
pts    = np.c_[xx.ravel(), yy.ravel()]

mu_grid, sig_grid = gp2.predict(pts, return_std=True)
ei_grid = np.array([expected_improvement(p, gp2, y_best) for p in pts])
mu_grid = mu_grid.reshape(80, 80)
ei_grid = ei_grid.reshape(80, 80)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Function 2 (2D) — GP Posterior and EI Acquisition Surface", fontsize=13, fontweight='bold')

c1 = axes[0].contourf(xx, yy, mu_grid, levels=30, cmap='viridis')
axes[0].scatter(*SUB1_X[2], c='red', s=120, zorder=5, label=f'Sub1 point (Y={SUB1_Y[2]:.4f})')
axes[0].set_title("GP Posterior Mean μ(x)")
axes[0].set_xlabel("x₁"); axes[0].set_ylabel("x₂")
axes[0].legend(fontsize=9)
plt.colorbar(c1, ax=axes[0])

c2 = axes[1].contourf(xx, yy, ei_grid, levels=30, cmap='plasma')
axes[1].scatter(*SUB1_X[2], c='red', s=120, zorder=5, label='Sub1 point')
axes[1].set_title("Expected Improvement EI(x) — ξ=0.02")
axes[1].set_xlabel("x₁"); axes[1].set_ylabel("x₂")
axes[1].legend(fontsize=9)
plt.colorbar(c2, ax=axes[1])

plt.tight_layout()
plt.savefig('f2_acquisition.png', dpi=120, bbox_inches='tight')
plt.show()
print("F2 acquisition surface saved to current directory.")


F2 acquisition surface saved to current directory.


## Step 7 — Optimise Acquisition Function

### Multi-Start L-BFGS-B + Differential Evolution

Since the acquisition surface can be multi-modal (multiple local maxima), we use:

1. **Multi-start L-BFGS-B**: fast gradient-based local optimisation from multiple
   starting points (exploitation candidates + LHS exploration grid)
2. **Differential Evolution** (for F1, F3, F5, F8): global stochastic optimiser,
   more robust for complex multi-modal landscapes in higher dimensions

### Boundary Constraints (`fix_dims`)

For functions where ARD confirmed dimension importance, we **hard-lock those
dimensions** by applying them after each gradient step. This is equivalent to
optimising in the subspace of free dimensions — computationally simple but
exactly respects the confirmed structure from Sub1.

```
x_candidate[fixed_dim] = confirmed_boundary_value  (applied at every evaluation)
```


In [7]:
def apply_fix(x, fix_dims):
    """Lock specified dimensions to their confirmed boundary values."""
    x = x.copy()
    for idx, val in fix_dims.items():
        x[idx] = val
    return x


def optimise_acquisition(func_id, gp, y_best):
    """
    Find the point that maximises the acquisition function for func_id.
    
    Strategy:
      1. Build candidate starts: exploitation (Gaussian around Sub1) + exploration (LHS)
      2. Apply fix_dims constraints
      3. Run L-BFGS-B from each start → take best
      4. Optionally run differential evolution for hard functions
    
    Returns
    -------
    best_x   : array (d,)  — proposed query point
    best_val : float        — acquisition value at best_x
    """
    d    = DIMS[func_id]
    t    = TACTICS[func_id]
    fix  = t['fix_dims']
    bnds = [(CLIP_LOW, CLIP_HIGH)] * d
    rng  = np.random.default_rng(func_id * 123)

    # ── Build candidate starts ──────────────────────────────────────────────
    starts = []

    # A) Exploitation: Gaussian perturbations around Sub1 best point
    for _ in range(t['n_exploit']):
        xp = np.clip(SUB1_X[func_id] + rng.normal(0, t['exploit_sigma'], d),
                     CLIP_LOW, CLIP_HIGH)
        starts.append(apply_fix(xp, fix))

    # B) Exploration: Latin Hypercube Sampling across full space
    n_e  = t['n_explore']
    cut  = np.linspace(0, 1, n_e + 1)
    lhs  = np.zeros((n_e, d))
    for j in range(d):
        idx = rng.permutation(n_e)
        lhs[:, j] = cut[idx] + rng.random(n_e) * (cut[1] - cut[0])
    for row in np.clip(lhs, CLIP_LOW, CLIP_HIGH):
        starts.append(apply_fix(row.copy(), fix))

    # ── Define objective (negated, since we minimise) ───────────────────────
    if t['use_ei']:
        obj = lambda x: -expected_improvement(x, gp, y_best)
    else:
        obj = lambda x: -upper_confidence_bound(x, gp, t['ucb_kappa'])

    best_val, best_x = -np.inf, SUB1_X[func_id].copy()

    # ── L-BFGS-B from all starts ─────────────────────────────────────────────
    for x0 in starts:
        try:
            res = minimize(obj, x0, method='L-BFGS-B', bounds=bnds,
                           options={'maxiter': 400, 'ftol': 1e-15})
            xr  = apply_fix(np.clip(res.x, CLIP_LOW, CLIP_HIGH), fix)
            val = -obj(xr)
            if val > best_val:
                best_val = val; best_x = xr.copy()
        except Exception:
            continue

    # ── Differential Evolution for complex landscapes ────────────────────────
    if func_id in [1, 3, 5, 8]:
        def de_obj(x):
            return obj(apply_fix(np.clip(x, CLIP_LOW, CLIP_HIGH), fix))
        try:
            res_de = differential_evolution(
                de_obj, bnds, seed=func_id, maxiter=200, tol=1e-10,
                popsize=15, mutation=(0.5, 1.2), recombination=0.9
            )
            xr  = apply_fix(np.clip(res_de.x, CLIP_LOW, CLIP_HIGH), fix)
            val = -obj(xr)
            if val > best_val:
                best_val = val; best_x = xr.copy()
        except Exception:
            pass

    return best_x, best_val


print("Acquisition optimiser defined.")
print("Acquisition strategy per function:")
for fid in range(1, 9):
    t   = TACTICS[fid]
    acq = f"EI ξ=0.02" if t['use_ei'] else f"UCB κ={t['ucb_kappa']}"
    lck = list(t['fix_dims'].keys())
    print(f"  F{fid}: {acq}  |  locked dims: {lck or 'none'}  "
          f"|  exploit σ={t['exploit_sigma']}")


Acquisition optimiser defined.
Acquisition strategy per function:
  F1: UCB κ=3.5  |  locked dims: none  |  exploit σ=0.35
  F2: EI ξ=0.02  |  locked dims: none  |  exploit σ=0.04
  F3: UCB κ=3.0  |  locked dims: none  |  exploit σ=0.4
  F4: EI ξ=0.02  |  locked dims: none  |  exploit σ=0.04
  F5: EI ξ=0.02  |  locked dims: [0, 2, 3]  |  exploit σ=0.06
  F6: EI ξ=0.02  |  locked dims: [3]  |  exploit σ=0.1
  F7: EI ξ=0.02  |  locked dims: [0]  |  exploit σ=0.05
  F8: EI ξ=0.02  |  locked dims: [3, 4]  |  exploit σ=0.06


## Step 8 — Main Optimisation Loop

Run the full pipeline for all 8 functions:
1. Build dataset (Sub1 confirmed point + structured prior)
2. Fit GP (Matérn 5/2, ARD)
3. Optimise acquisition function
4. Format and store Submission 2 string


In [9]:
def fmt(x):
    """Format a candidate point as the required portal submission string."""
    return "-".join(f"{np.clip(v, CLIP_LOW, CLIP_HIGH):.6f}" for v in x)


# ─── Main loop ────────────────────────────────────────────────────────────────
results = {}

print("=" * 80)
print("  BBO CAPSTONE — SUBMISSION 2 | FULL PIPELINE RUN")
print("=" * 80)

for fid in range(1, 9):
    d      = DIMS[fid]
    y1     = SUB1_Y[fid]
    yi     = INIT_BEST_Y[fid]
    y_best = max(y1, yi)
    delta  = y1 - yi
    status = "IMPROVED ✓" if delta > 0 else ("DEGRADED ✗" if delta < -1e-10 else "UNCHANGED")

    # Step A: dataset
    X_tr, Y_tr = datasets[fid]       # already built in Step 5

    # Step B: GP (already fitted in Step 5)
    gp = gps[fid]

    # Step C: optimise acquisition
    x_new, acq_val = optimise_acquisition(fid, gp, y_best)

    # Step D: GP prediction at proposed point
    mu_new, sig_new = gp.predict(x_new.reshape(1,-1), return_std=True)
    mu_val  = float(np.squeeze(mu_new))
    sig_val = float(np.squeeze(sig_new))
    sub_str = fmt(x_new)

    acq_label  = 'EI xi=0.02' if TACTICS[fid]['use_ei'] else 'UCB kappa=' + str(TACTICS[fid]['ucb_kappa'])
    fixed_dims = list(TACTICS[fid]['fix_dims'].keys())
    fixed_str  = str(fixed_dims) if fixed_dims else 'none'
    print(f"\n  F{fid} | d={d} | Sub1_Y={y1:.5g} | InitBest={yi:.5g} | Delta={delta:+.4g} | {status}")
    print(f"     Acquisition : {acq_label}")
    print(f"     Fixed dims  : {fixed_str}")
    print(f"     Proposed X  : {np.round(x_new, 6)}")
    print(f"     GP mu / sig : {mu_val:.5g} / {sig_val:.5g}")
    print(f"     SUBMISSION  : {sub_str}")

    results[fid] = {
        'dims': d, 'sub1_y': y1, 'init_best_y': yi, 'delta': delta,
        'x_new': x_new.tolist(), 'gp_mu': mu_val, 'gp_sigma': sig_val,
        'submission_string': sub_str
    }

print(f"\n{'='*80}")
print("  SUBMISSION 2 — FINAL STRINGS (copy-paste to portal)")
print(f"{'='*80}")
for fid, r in results.items():
    print(f"  F{fid} (d={r['dims']}): {r['submission_string']}")


  BBO CAPSTONE — SUBMISSION 2 | FULL PIPELINE RUN

  F1 | d=2 | Sub1_Y=-2.4675e-270 | InitBest=0 | Delta=-2.467e-270 | UNCHANGED
     Acquisition : UCB kappa=3.5
     Fixed dims  : none
     Proposed X  : [0.999999 0.999999]
     GP mu / sig : -1.5944e-272 / 0.0021582
     SUBMISSION  : 0.999999-0.999999

  F2 | d=2 | Sub1_Y=0.72374 | InitBest=0.6112 | Delta=+0.1125 | IMPROVED ✓
     Acquisition : EI xi=0.02
     Fixed dims  : none
     Proposed X  : [0.698486 0.      ]
     GP mu / sig : 0.63471 / 0.14126
     SUBMISSION  : 0.698486-0.000000

  F3 | d=3 | Sub1_Y=-0.08912 | InitBest=-0.034835 | Delta=-0.05428 | DEGRADED ✗
     Acquisition : UCB kappa=3.0
     Fixed dims  : none
     Proposed X  : [0.850892 0.035316 0.936193]
     GP mu / sig : -0.0016233 / 0.023855
     SUBMISSION  : 0.850892-0.035316-0.936193

  F4 | d=4 | Sub1_Y=0.25958 | InitBest=-4.0255 | Delta=+4.285 | IMPROVED ✓
     Acquisition : EI xi=0.02
     Fixed dims  : none
     Proposed X  : [0.999999 0.       0.       0

## Step 9 — Post-Analysis: Strategy Comparison Sub1 vs Sub2


In [10]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("BBO Submission 2 — Per-Function GP Uncertainty at Sub1 vs Proposed Point",
             fontsize=13, fontweight='bold')

axes = axes.ravel()

for i, fid in enumerate(range(1, 9)):
    gp     = gps[fid]
    x1     = SUB1_X[fid]
    x2     = np.array(results[fid]['x_new'])
    d      = DIMS[fid]

    # Generate a 1D slice: interpolate from x1 to x2
    alphas = np.linspace(0, 1, 60)
    X_line = np.array([(1-a)*x1 + a*x2 for a in alphas])
    mu_line, sig_line = gp.predict(X_line, return_std=True)

    ax = axes[i]
    ax.fill_between(alphas,
                    mu_line - 1.96*sig_line,
                    mu_line + 1.96*sig_line,
                    alpha=0.3, color='steelblue', label='95% CI')
    ax.plot(alphas, mu_line, 'b-', lw=2, label='GP mean')
    ax.axvline(0, color='red',   ls='--', lw=1.5, label=f'Sub1 Y={SUB1_Y[fid]:.3g}')
    ax.axvline(1, color='green', ls='--', lw=1.5, label=f'Sub2 (proposed)')
    ax.set_title(f"F{fid} (d={d}) | Δ={results[fid]['delta']:+.3g}", fontsize=10)
    ax.set_xlabel("α (0=Sub1, 1=Sub2 proposed)")
    ax.set_ylabel("GP posterior")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sub2_gp_slices.png', dpi=120, bbox_inches='tight')
plt.show()
print("GP slice plots saved to current directory.")


GP slice plots saved to current directory.


In [11]:
# ─── Final summary table ─────────────────────────────────────────────────────
print(f"\n{'='*100}")
print(f"  {'F':>2} | {'d':>2} | {'Sub1 Y':>14} | {'Init Best':>10} | {'ΔY':>8}"
      f" | {'GP μ(x*)':>10} | {'GP σ':>8} | Submission 2 String")
print("─" * 100)
for fid in range(1, 9):
    r = results[fid]
    print(f"  {fid:>2} | {r['dims']:>2} | {r['sub1_y']:>14.5g} | "
          f"{r['init_best_y']:>10.5g} | {r['delta']:>+8.4g} | "
          f"{r['gp_mu']:>10.5g} | {r['gp_sigma']:>8.5g} | {r['submission_string']}")
print("─" * 100)
print("\nAll 8 Submission 2 query strings generated successfully.")



   F |  d |         Sub1 Y |  Init Best |       ΔY |   GP μ(x*) |     GP σ | Submission 2 String
────────────────────────────────────────────────────────────────────────────────────────────────────
   1 |  2 |   -2.4675e-270 |          0 | -2.467e-270 | -1.5944e-272 | 0.0021582 | 0.999999-0.999999
   2 |  2 |        0.72374 |     0.6112 |  +0.1125 |    0.63471 |  0.14126 | 0.698486-0.000000
   3 |  3 |       -0.08912 |  -0.034835 | -0.05428 | -0.0016233 | 0.023855 | 0.850892-0.035316-0.936193
   4 |  4 |        0.25958 |    -4.0255 |   +4.285 |    0.16336 | 0.067513 | 0.999999-0.000000-0.000000-0.365908
   5 |  4 |         2105.9 |     1088.9 |    +1017 |     1134.7 |   517.12 | 0.000000-0.000000-0.999999-0.999999
   6 |  5 |       -0.55077 |   -0.71427 |  +0.1635 |  -0.034668 |  0.15852 | 0.142734-0.321812-0.416483-0.999999-0.304415
   7 |  6 |         2.2073 |      1.365 |  +0.8423 |     1.5588 |  0.47872 | 0.000000-0.302741-0.000000-0.187177-0.000000-0.167183
   8 |  8 |         9.

## Summary

### What changed from Submission 1 → Submission 2

| Function | Sub1 outcome | Sub2 strategy | Key change |
|---|---|---|---|
| F1 | ≈ 0 (flat) | UCB κ=3.5 | Opposite corner exploration |
| F2 | +0.11 gain | EI tight exploit | Keep x1≈0.70; push x2→0 |
| F3 | −0.054 degraded | UCB κ=3.0 | Full exploration away from Sub1 region |
| F4 | +4.29 gain | EI exploit | Test boundary extension of confirmed basin |
| F5 | +1017 gain | EI + fix x1=0, x3=x4=1 | Sweep x2 toward 0 boundary |
| F6 | +0.16 gain | EI + fix x4=1 | Unlock x5 from 0; explore x1–x3 |
| F7 | +0.84 gain | EI + fix x1=0 | Tighten exploitation in remaining dims |
| F8 | +0.26 gain | EI + fix x4=0, x5=1 | Push x8→1; adjust x2, x3 |

### Next Steps (Submission 3)
- Append Sub2 true-Y feedback to confirmed dataset
- Re-fit GP with now 2 confirmed points per function
- Use Kriging variance maps to identify the highest-uncertainty regions
- Consider Thompson Sampling for functions where EI has plateaued
